In [0]:
#pip install langgraph

In [0]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Any

# Define o estado que o agente vai carregar durante a execução
class AgentState(TypedDict):
    data: Any
    model_results: Any
    validation: Any
    status: str

workflow = StateGraph(AgentState)

# Adiciona os nós baseados nas ferramentas que criamos
workflow.add_node("feature_engineering", lambda state: {"data": tool_feature_engineering_and_split(), "status": "engineered"})
workflow.add_node("training", lambda state: {"model_results": tool_hyperparameter_tuning_and_train(state["data"]), "status": "trained"})
workflow.add_node("validation", lambda state: {"validation": tool_rigorous_validation(state["model_results"], state["data"]), "status": "validated"})
workflow.add_node("deployment", lambda state: {"deploy": tool_register_and_deploy_model(state["model_results"], state["validation"]), "status": "deployed"})

# Define o fluxo lógico e as decisões condicionais do agente
workflow.set_entry_point("feature_engineering")
workflow.add_edge("feature_engineering", "training")
workflow.add_edge("training", "validation")

# Decisão inteligente: Se aprovado, vai para deploy; se reprovado, encerra ou ajusta
def should_deploy(state):
    if state["validation"]["approved"]:
        return "deploy"
    else:
        return "stop"

workflow.add_conditional_edges(
    "validation",
    should_deploy,
    {
        "deploy": "deployment",
        "stop": END
    }
)
workflow.add_edge("deployment", END)

app = workflow.compile()
print("[Agente] LangGraph compilado e pronto para execução autônoma.")